In [21]:
import pandas as pd
import numpy as np
from pathlib import Path

# =========================
# PATHS
# =========================
BASE_DIR = Path.cwd()
if BASE_DIR.name == "scripts":
    BASE_DIR = BASE_DIR.parent

input_path = BASE_DIR / "data/factsheet/NFHS_5_India_Districts_Factsheet_Data.xls"
output_path = BASE_DIR / "data/variables/nfhs_ncd_pct.csv"

print("Input:", input_path)
print("Exists:", input_path.exists())

# =========================
# LOAD DATA
# =========================
df = pd.read_excel(input_path)

# =========================
# AUTO-DETECT DISTRICT COLUMN
# =========================
district_candidates = [c for c in df.columns if "district" in str(c).lower()]
if not district_candidates:
    raise ValueError("No district column found")

district_col = district_candidates[0]
print("Using district column:", district_col)

# =========================
# CLEAN DISTRICT VALUES
# =========================
df[district_col] = (
    df[district_col]
    .astype(str)
    .str.strip()
    .str.replace("\n", "", regex=True)
    .str.replace("\t", "", regex=True)
    .str.replace("\xa0", "", regex=True)
)

# =========================
# NFHS → STANDARD DISTRICT MAP
# =========================
district_map = {
    "Anugul": "Anugul",
    "Angul": "Anugul",
    "Balangir": "Balangir",
    "Baleshwar": "Baleshwar",
    "Balasore": "Baleshwar",
    "Bargarh": "Bargarh",
    "Bhadrak": "Bhadrak",
    "Boudh": "Boudh",
    "Baudh": "Boudh",
    "Cuttack": "Cuttack",
    "Debagarh": "Deogarh",
    "Deogarh": "Deogarh",
    "Dhenkanal": "Dhenkanal",
    "Gajapati": "Gajapati",
    "Ganjam": "Ganjam",
    "Jagatsinghapur": "Jagatsinghapur",
    "Jajapur": "Jajapur",
    "Jharsuguda": "Jharsuguda",
    "Kalahandi": "Kalahandi",
    "Kandhamal": "Kandhamal",
    "Kendrapara": "Kendrapara",
    "Kendujhar": "Kendujhar",
    "Keonjhar": "Kendujhar",
    "Khordha": "Khordha",
    "Koraput": "Koraput",
    "Malkangiri": "Malkangiri",
    "Mayurbhanj": "Mayurbhanj",
    "Nabarangapur": "Nabarangpur",
    "Nabarangpur": "Nabarangpur",
    "Nayagarh": "Nayagarh",
    "Nuapada": "Nuapada",
    "Puri": "Puri",
    "Rayagada": "Rayagada",
    "Sambalpur": "Sambalpur",
    "Sonepur": "Sonepur",
    "Subarnapur": "Sonepur",
    "Sundargarh": "Sundargarh"
}

df["district_std"] = df[district_col].replace(district_map)

# =========================
# TARGET DISTRICTS
# =========================
target_districts = [
    "Anugul","Balangir","Baleshwar","Bargarh","Bhadrak","Boudh","Cuttack",
    "Deogarh","Dhenkanal","Gajapati","Ganjam","Jagatsinghapur","Jajapur",
    "Jharsuguda","Kalahandi","Kandhamal","Kendrapara","Kendujhar","Khordha",
    "Koraput","Malkangiri","Mayurbhanj","Nabarangpur","Nayagarh","Nuapada",
    "Puri","Rayagada","Sambalpur","Sonepur","Sundargarh"
]

# =========================
# FILTER ODISHA
# =========================
df_odisha = df[df["district_std"].isin(target_districts)].copy()

# =========================
# INDICATOR COLUMNS
# =========================
cols = {
    "women_sugar": "Women age 15 years and above wih very high (>160 mg/dl) Blood sugar level23 (%)",
    "men_sugar": "Men (age 15 years and above wih  very high (>160 mg/dl) Blood sugar level23 (%)",
    "women_bp": "Women age 15 years and above wih Moderately or severely elevated blood pressure (Systolic ≥160 mm of Hg and/or Diastolic ≥100 mm of Hg) (%)",
    "men_bp": "Men age 15 years and above wih Moderately or severely elevated blood pressure (Systolic ≥160 mm of Hg and/or Diastolic ≥100 mm of Hg) (%)"
}

# =========================
# VALIDATE COLUMNS
# =========================
missing_cols = [c for c in cols.values() if c not in df_odisha.columns]
if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")

# =========================
# CLEAN NUMERIC VALUES
# =========================
for c in cols.values():
    df_odisha[c] = pd.to_numeric(df_odisha[c], errors="coerce")

# =========================
# COMPUTE MEAN NCD INDEX
# =========================
df_odisha["pct_ncd"] = df_odisha[list(cols.values())].mean(axis=1)

# =========================
# FINAL OUTPUT (ALL 4 + MEAN)
# =========================
out = df_odisha[["district_std"] + list(cols.values()) + ["pct_ncd"]].copy()

out = out.rename(columns={
    "district_std": "district",
    cols["women_sugar"]: "women_sugar",
    cols["men_sugar"]: "men_sugar",
    cols["women_bp"]: "women_bp",
    cols["men_bp"]: "men_bp"
})

# enforce order
out["district"] = pd.Categorical(out["district"], categories=target_districts, ordered=True)
out = out.sort_values("district")

# =========================
# VALIDATION
# =========================
matched = out["district"].nunique()
missing = set(target_districts) - set(out["district"].dropna())

print("\nMatched districts:", matched)
print("Missing districts:", missing)

assert matched == 30, "ERROR: Not all 30 districts matched!"

# =========================
# SAVE OUTPUT
# =========================
output_path.parent.mkdir(parents=True, exist_ok=True)
out.to_csv(output_path, index=False)

print("\nSaved to:", output_path)
print(out.head())

Input: /home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/data_extractor/nfhs/data/factsheet/NFHS_5_India_Districts_Factsheet_Data.xls
Exists: True
Using district column: District

Matched districts: 30
Missing districts: set()

Saved to: /home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/data_extractor/nfhs/data/variables/nfhs_ncd_pct.csv
      district  women_sugar  men_sugar  women_bp  men_bp  pct_ncd
449     Anugul         6.48       9.05      3.65    4.58   5.9400
458   Balangir         4.65       6.58      2.70    3.45   4.3450
442  Baleshwar         5.12       7.39      5.94    6.60   6.2625
435    Bargarh         7.80       8.30      5.07    4.77   6.4850
443    Bhadrak         5.84       9.24      5.08    5.88   6.5100


/tmp/ipykernel_46771/2616739361.py:88: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["district_std"] = df[district_col].replace(district_map)


In [10]:
import sys
print(sys.executable)

!{sys.executable} -m ensurepip --upgrade
!{sys.executable} -m pip install --upgrade pip
!{sys.executable} -m pip install xlrd

/home/root_1/Documents/CDL/repos/IDS-DRR_Heat/Heat-odisha/.venv/bin/python
Looking in links: /tmp/tmper1loo60
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 2.1 MB/s eta 0:00:0000:0100:010m
  Attempting uninstall: pip
    Found existing installation: pip 24.0
    Uninstalling pip-24.0:
      Successfully uninstalled pip-24.0
